<a href="https://colab.research.google.com/github/joaocanaslopes/Exercises_AVD/blob/main/ex5_avd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import zipfile
import seaborn as sns # For plotting
import matplotlib.pyplot as plt # For showing plots
import numpy as np

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/isa-ulisboa/greends-avcad-2026/main/examples/EFIplus_medit.zip"

df = pd.read_csv(url, compression="zip", sep=";")

df.head()

,Site_code,Latitude,Longitude,Country,Catchment_name,Galiza,Subsample,Calib_EFI_Medit,Calib_connect,Calib_hydrol,...,Squalius malacitanus,Squalius pyrenaicus,Squalius torgalensis,Thymallus thymallus,Tinca tinca,Zingel asper,Squalius sp,Barbatula sp,Phoxinus sp,Iberochondrostoma_sp
0,ES_01_0002,38.102003,-4.096070,Spain,Guadalquivir,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,ES_02_0001,40.530188,-1.887796,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,ES_02_0002,40.595432,-1.928079,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,ES_02_0003,40.656184,-1.989831,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
4,ES_02_0004,40.676402,-2.036274,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
print(df.columns.tolist())

['Site_code', 'Latitude', 'Longitude', 'Country', 'Catchment_name', 'Galiza', 'Subsample', 'Calib_EFI_Medit', 'Calib_connect', 'Calib_hydrol', 'Calib_morphol', 'Calib_wqual', 'Geomorph1', 'Geomorph2', 'Geomorph3', 'Water_source_type', 'Flow_regime', 'Altitude', 'Geological_typology', 'Actual_river_slope', 'Natural_sediment', 'Elevation_mean_catch', 'prec_ann_catch', 'temp_ann', 'temp_jan', 'temp_jul', 'Barriers_catchment_down', 'Barriers_river_segment_up', 'Barriers_river_segment_down', 'Barriers_number_river_segment_up', 'Barriers_number_river_segment_down', 'Barriers_distance_river_segment_up', 'Barriers_distance_river_segment_down', 'Impoundment', 'Hydropeaking', 'Water_abstraction', 'Hydro_mod', 'Temperature_impact', 'Velocity_increase', 'Reservoir_flushing', 'Sedimentation', 'Channelisation', 'Cross_sec', 'Instream_habitat', 'Riparian_vegetation', 'Embankment', 'Floodprotection', 'Floodplain', 'Toxic_substances', 'Acidification', 'Water_quality_index', 'Eutrophication', 'Organic_p

From the column list, I've identified the relevant columns as `temp_ann` for 'Mean Annual Temperature' and `Salmo trutta fario` for the presence/absence of Brown Trout.

In [ ]:
from scipy import stats
import numpy as np

# Define the columns
temperature_col = 'temp_ann' # Corrected column name
fish_presence_col = 'Salmo trutta fario'

# Separate data into presence and absence groups
presence_temp = df[df[fish_presence_col] == 1][temperature_col].dropna()
absence_temp = df[df[fish_presence_col] == 0][temperature_col].dropna()

print(f"Number of presence sites: {len(presence_temp)}")
print(f"Number of absence sites: {len(absence_temp)}")

# --- Non-standardized Values ---
print("\n--- Non-standardized Values ---")

# Independent t-test (for means)
# Null Hypothesis (H0): The mean annual temperature is equal between presence and absence sites.
t_stat_non_std, p_val_t_non_std = stats.ttest_ind(presence_temp, absence_temp, equal_var=False) # Welch's t-test assuming unequal variances
print(f"T-test (non-standardized):\n  Null Hypothesis (H0): Mean annual temperature is equal for presence and absence sites.\n  t-statistic: {t_stat_non_std:.3f}, p-value: {p_val_t_non_std:.3f}")

# Mann-Whitney U test (for medians/distributions)
# Null Hypothesis (H0): The distribution of annual temperature is the same for presence and absence sites.
# (Alternatively, the probability of a randomly selected temperature from the presence group being greater than a randomly selected temperature from the absence group is 0.5)
rank_stat_non_std, p_val_mw_non_std = stats.mannwhitneyu(presence_temp, absence_temp, alternative='two-sided')
print(f"Mann-Whitney U test (non-standardized):\n  Null Hypothesis (H0): The distribution of annual temperature is the same for presence and absence sites.\n  U-statistic: {rank_stat_non_std:.3f}, p-value: {p_val_mw_non_std:.3f}")

# --- Standardized Values ---
print("\n--- Standardized Values ---")

# Standardize the 'temp_ann' column (Z-score normalization)
mean_temp = df[temperature_col].mean()
std_temp = df[temperature_col].std()
df['Temp_Ann_Mean_standardized'] = (df[temperature_col] - mean_temp) / std_temp

# Separate standardized data into presence and absence groups
presence_temp_std = df[df[fish_presence_col] == 1]['Temp_Ann_Mean_standardized'].dropna()
absence_temp_std = df[df[fish_presence_col] == 0]['Temp_Ann_Mean_standardized'].dropna()

# Independent t-test (for means) on standardized values
# Null Hypothesis (H0): The mean *standardized* annual temperature is equal between presence and absence sites.
t_stat_std, p_val_t_std = stats.ttest_ind(presence_temp_std, absence_temp_std, equal_var=False)
print(f"T-test (standardized):\n  Null Hypothesis (H0): Mean standardized annual temperature is equal for presence and absence sites.\n  t-statistic: {t_stat_std:.3f}, p-value: {p_val_t_std:.3f}")

# Mann-Whitney U test (for medians/distributions) on standardized values
# Null Hypothesis (H0): The distribution of *standardized* annual temperature is the same for presence and absence sites.
rank_stat_std, p_val_mw_std = stats.mannwhitneyu(presence_temp_std, absence_temp_std, alternative='two-sided')
print(f"Mann-Whitney U test (standardized):\n  Null Hypothesis (H0): The distribution of standardized annual temperature is the same for presence and absence sites.\n  U-statistic: {rank_stat_std:.3f}, p-value: {p_val_mw_std:.3f}")

# --- Comparison of Results ---
print("\n--- Comparison of Results ---")
print("The p-values for both the t-test and Mann-Whitney U test are identical for non-standardized and standardized values. This is expected because standardization (z-score normalization) is a linear transformation and does not change the shape of the distribution, nor the relative distances between values, thus not affecting the p-value of these comparative tests. If the p-value is below a chosen significance level (e.g., 0.05), we reject the null hypothesis, suggesting a significant difference in temperature between presence and absence sites. Otherwise, we fail to reject the null hypothesis.")

Number of presence sites: 2941
Number of absence sites: 1900

--- Non-standardized Values ---
T-test (non-standardized):
  Null Hypothesis (H0): Mean annual temperature is equal for presence and absence sites.
  t-statistic: -43.960, p-value: 0.000
Mann-Whitney U test (non-standardized):
  Null Hypothesis (H0): The distribution of annual temperature is the same for presence and absence sites.
  U-statistic: 1027812.500, p-value: 0.000

--- Standardized Values ---
T-test (standardized):
  Null Hypothesis (H0): Mean standardized annual temperature is equal for presence and absence sites.
  t-statistic: -43.960, p-value: 0.000
Mann-Whitney U test (standardized):
  Null Hypothesis (H0): The distribution of standardized annual temperature is the same for presence and absence sites.
  U-statistic: 1027812.500, p-value: 0.000

--- Comparison of Results ---
The p-values for both the t-test and Mann-Whitney U test are identical for non-standardized and standardized values. This is expected beca

### Interpretação dos Testes de Temperatura

**Resultados dos Testes para Valores Não-Padronizados e Padronizados:**

*   **Teste t (não-padronizado e padronizado):** O t-statistic de aproximadamente -43.960 com um p-value de 0.000 para ambos. Isto significa que **rejeitamos a hipótese nula** de que a temperatura média anual é igual entre os locais com e sem *Salmo trutta fario*. Há uma diferença estatisticamente significativa nas temperaturas médias.

*   **Teste U de Mann-Whitney (não-padronizado e padronizado):** O U-statistic de 1027812.500 com um p-value de 0.000 para ambos. Isto significa que **rejeitamos a hipótese nula** de que a distribuição da temperatura anual é a mesma para os locais com e sem *Salmo trutta fario*. Há uma diferença estatisticamente significativa nas distribuições de temperatura.

**Conclusão sobre a Padronização:**

Os p-valores para ambos os testes (t-test e Mann-Whitney U test) são idênticos para os valores não-padronizados e padronizados. Isso ocorre porque a padronização (normalização Z-score) é uma transformação linear. Ela não altera a forma da distribuição dos dados nem as distâncias relativas entre os valores, e, portanto, não afeta o resultado dos testes comparativos como o t-test e o Mann-Whitney U test em termos de significância estatística. Ambos os testes indicam robustamente que a temperatura média anual (ou sua distribuição) é significativamente diferente nos locais onde *Salmo trutta fario* está presente em comparação com os locais onde está ausente.

### Teste Qui-quadrado de Independência

Vamos realizar um teste Qui-quadrado para verificar se a frequência de sítios com presença e ausência de *Salmo trutta fario* é independente do país.

**Hipótese Nula (H0):** A presença ou ausência de *Salmo trutta fario* é independente do país (não há associação entre a ocorrência da espécie e o país).

**Hipótese Alternativa (H1):** A presença ou ausência de *Salmo trutta fario* não é independente do país (existe uma associação entre a ocorrência da espécie e o país).

In [ ]:
from scipy.stats import chi2_contingency

# Criar uma tabela de contingência entre 'Country' e 'Salmo trutta fario'
contingency_table = pd.crosstab(df['Country'], df['Salmo trutta fario'])

print("Tabela de Contingência:")
display(contingency_table)

# Realizar o teste Qui-quadrado de independência
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"\nResultados do Teste Qui-quadrado:")
print(f"Estatística Qui-quadrado: {chi2:.3f}")
print(f"Valor p: {p_value:.3f}")
print(f"Graus de Liberdade (dof): {dof}")
print("Valores Esperados (se H0 fosse verdadeira):")
display(pd.DataFrame(expected, index=contingency_table.index, columns=contingency_table.columns))

Tabela de Contingência:


Salmo trutta fario,0,1
Country,,
France,13,59
Italy,109,76
Portugal,615,252
Spain,1239,2648



Resultados do Teste Qui-quadrado:
Estatística Qui-quadrado: 496.372
Valor p: 0.000
Graus de Liberdade (dof): 3
Valores Esperados (se H0 fosse verdadeira):


Salmo trutta fario,0,1
Country,,
France,28.391938,43.608062
Italy,72.951507,112.048493
Portugal,341.886250,525.113750
Spain,1532.770305,2354.229695


### Interpretação dos Resultados do Teste Qui-quadrado

**Estatística Qui-quadrado:** 496.372
**Valor p:** 0.000
**Graus de Liberdade (dof):** 3

Com um valor p de 0.000 (muito menor que qualquer nível de significância comum, como 0.05), **rejeitamos a hipótese nula (H0)**. Isso significa que há uma **associação estatisticamente significativa** entre a presença ou ausência de *Salmo trutta fario* e o país. Em outras palavras, a ocorrência da espécie não é independente do país; a frequência de sítios com e sem a espécie varia significativamente entre os diferentes países.

### Visualização com Gráfico Aluvial

Para visualizar a relação entre o país e a presença/ausência da espécie, usaremos um gráfico aluvial (Sankey Diagram) utilizando a biblioteca `plotly`.

In [ ]:
import plotly.graph_objects as go

# Prepare data for Plotly Sankey diagram
# Step 1: Create a list of unique labels for all nodes (countries and presence/absence)
labels = []
labels.extend(alluvial_data['Country'].unique())
labels.extend(alluvial_data['Salmo trutta fario'].unique())

# Step 2: Create a mapping from label to its index
label_to_index = {label: i for i, label in enumerate(labels)}

# Step 3: Prepare the links data
source = [label_to_index[country] for country in alluvial_data['Country']]
target = [label_to_index[status] for status in alluvial_data['Salmo trutta fario']]
value = alluvial_data['count'].tolist()

# Step 4: Create the Sankey diagram
fig = go.Figure(data=[
    go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=labels,
            # You can add colors for nodes if desired, e.g., using a color list
        ),
        link=dict(
            source=source, # indices correspond to labels, e.g., 0 for 'France'
            target=target, # indices correspond to labels, e.g., 4 for 'Ausente'
            value=value,   # sums over the values of all links
        )
    )
])

fig.update_layout(
    title_text="Relação entre País e Presença/Ausência de Salmo trutta fario",
    font_size=10
)

fig.show()

Este diagrama de Sankey do Plotly representa visualmente o fluxo de países para a presença ou ausência de *Salmo trutta fario*. Cada nó é um país ou um status de presença/ausência, e a largura das ligações indica a contagem de sítios para essa combinação específica. Esta visualização interativa permite-nos compreender rapidamente a distribuição da espécie em diferentes países.

### Teste de Diferença na Elevação Média em Bacias Hidrográficas

Vamos testar se existem diferenças significativas na elevação média (`Elevation_mean_catch`) entre as oito bacias hidrográficas mais amostradas. Assumiremos que a elevação média é normalmente distribuída.

**Hipóteses Nulas:**

*   **Para o teste ANOVA (F-test):** A hipótese nula (H0) é que a elevação média na bacia hidrográfica (`Elevation_mean_catch`) é a mesma em todas as oito bacias mais amostradas.
*   **Para os testes Post-Hoc (Tukey HSD):** Para cada par de bacias hidrográficas, a hipótese nula (H0) é que a elevação média é a mesma entre essas duas bacias.

In [ ]:
import pandas as pd
from scipy import stats

# 1. Identify the eight most sampled catchments (ensure this is still needed or already defined)
# This was previously defined, so we'll just use it.

# 2. Filter the DataFrame to include only these catchments
df_filtered = df[df['Catchment_name'].isin(most_sampled_catchments)].copy()

# Prepare data for ANOVA: extract elevation data for each catchment
# Drop NaNs to ensure all arrays are clean for ANOVA
catchment_elevation_data = []
for catchment in most_sampled_catchments:
    catchment_elevation_data.append(df_filtered[df_filtered['Catchment_name'] == catchment]['Elevation_mean_catch'].dropna())

# 3. Perform ANOVA test
# Null Hypothesis (H0): The mean Elevation_mean_catch is equal across all eight most sampled catchments.
# Alternative Hypothesis (H1): At least one catchment's mean Elevation_mean_catch is different.
f_statistic, p_value_anova = stats.f_oneway(*catchment_elevation_data)

print(f"\n--- Resultados do Teste ANOVA ---")
print(f"Estatística F: {f_statistic:.3f}")
print(f"Valor p: {p_value_anova:.3e}")

alpha = 0.05
if p_value_anova < alpha:
    print(f"O valor p ({p_value_anova:.3e}) é menor que alpha ({alpha}). Rejeitamos a hipótese nula.")
    print("Existem diferenças significativas na elevação média entre as oito bacias hidrográficas mais amostradas.")
else:
    print(f"O valor p ({p_value_anova:.3e}) é maior ou igual a alpha ({alpha}). Não rejeitamos a hipótese nula.")
    print("Não há diferenças significativas na elevação média entre as oito bacias hidrográficas mais amostradas.")


--- Resultados do Teste ANOVA ---
Estatística F: 227.954
Valor p: 1.370e-285
O valor p (1.370e-285) é menor que alpha (0.05). Rejeitamos a hipótese nula.
Existem diferenças significativas na elevação média entre as oito bacias hidrográficas mais amostradas.


### Teste Post-Hoc de Dunn (após Kruskal-Wallis)

O Teste de Dunn é uma opção comum para identificar quais pares de grupos diferem significativamente após um resultado significativo do Kruskal-Wallis.

Para isso, usaremos a implementação do teste de Dunn que faz parte da biblioteca `statsmodels` (ou uma implementação manual se necessário). Para `statsmodels`, o teste é chamado `posthoc_dunn` e está na biblioteca `scikit-posthocs`, que não está instalada. Vamos instalar primeiro.

In [ ]:
!pip install scikit-posthocs

In [ ]:
import scikit_posthocs as sp

# Prepare the data as a DataFrame, dropping NaNs first
# Ensure df_filtered_tukey_check contains the original columns for group and values
dunn_df = df_filtered_tukey_check[['Elevation_mean_catch', 'Catchment_name']].dropna()

# Realizar o teste de Dunn usando val_col e group_col
dunn_results = sp.posthoc_dunn(
    a=dunn_df,
    val_col='Elevation_mean_catch',
    group_col='Catchment_name',
    p_adjust='bonferroni'
)

print("\n--- Resultados do Teste Post-Hoc de Dunn (com correção Bonferroni) ---")
display(dunn_results)

# Interpretação dos resultados
alpha = 0.05
print(f"\nPares de bacias hidrográficas com diferenças significativas (p < {alpha}):")

significant_dunn_pairs = []
for i in range(len(dunn_results.index)):
    for j in range(i + 1, len(dunn_results.columns)):
        group1 = dunn_results.index[i]
        group2 = dunn_results.columns[j]
        p_val = dunn_results.iloc[i, j]
        if p_val < alpha:
            significant_dunn_pairs.append(f"{group1} vs {group2} (p={p_val:.3e})")

if significant_dunn_pairs:
    for pair in significant_dunn_pairs:
        print(f"- {pair}")
else:
    print("Nenhum par de bacias hidrográficas mostrou diferença significativa após a correção Bonferroni para múltiplas comparações.")



--- Resultados do Teste Post-Hoc de Dunn (com correção Bonferroni) ---


,Cantabrica,Catala,Douro,Ebro,Galiza-Norte,Guadia,Minho,Tejo
Cantabrica,1.000000e+00,7.776285e-01,1.317420e-31,8.151841e-69,1.060456e-13,3.075474e-07,1.737715e-35,9.271577e-09
Catala,7.776285e-01,1.000000e+00,3.884022e-13,1.628849e-29,3.595515e-16,1.742541e-10,6.508382e-13,1.280597e-01
Douro,1.317420e-31,3.884022e-13,1.000000e+00,2.309299e-03,7.926606e-96,3.372119e-60,1.000000e+00,1.473248e-08
Ebro,8.151841e-69,1.628849e-29,2.309299e-03,1.000000e+00,6.749938e-189,1.457688e-106,2.625631e-07,1.605385e-28
Galiza-Norte,1.060456e-13,3.595515e-16,7.926606e-96,6.749938e-189,1.000000e+00,1.000000e+00,7.422061e-122,5.041730e-51
Guadia,3.075474e-07,1.742541e-10,3.372119e-60,1.457688e-106,1.000000e+00,1.000000e+00,9.038610e-68,1.769566e-29
Minho,1.737715e-35,6.508382e-13,1.000000e+00,2.625631e-07,7.422061e-122,9.038610e-68,1.000000e+00,2.084380e-08
Tejo,9.271577e-09,1.280597e-01,1.473248e-08,1.605385e-28,5.041730e-51,1.769566e-29,2.084380e-08,1.000000e+00



Pares de bacias hidrográficas com diferenças significativas (p < 0.05):
- Cantabrica vs Douro (p=1.317e-31)
- Cantabrica vs Ebro (p=8.152e-69)
- Cantabrica vs Galiza-Norte (p=1.060e-13)
- Cantabrica vs Guadia (p=3.075e-07)
- Cantabrica vs Minho (p=1.738e-35)
- Cantabrica vs Tejo (p=9.272e-09)
- Catala vs Douro (p=3.884e-13)
- Catala vs Ebro (p=1.629e-29)
- Catala vs Galiza-Norte (p=3.596e-16)
- Catala vs Guadia (p=1.743e-10)
- Catala vs Minho (p=6.508e-13)
- Douro vs Ebro (p=2.309e-03)
- Douro vs Galiza-Norte (p=7.927e-96)
- Douro vs Guadia (p=3.372e-60)
- Douro vs Tejo (p=1.473e-08)
- Ebro vs Galiza-Norte (p=6.750e-189)
- Ebro vs Guadia (p=1.458e-106)
- Ebro vs Minho (p=2.626e-07)
- Ebro vs Tejo (p=1.605e-28)
- Galiza-Norte vs Minho (p=7.422e-122)
- Galiza-Norte vs Tejo (p=5.042e-51)
- Guadia vs Minho (p=9.039e-68)
- Guadia vs Tejo (p=1.770e-29)
- Minho vs Tejo (p=2.084e-08)


### Teste H de Kruskal-Wallis (Equivalente Não Paramétrico do ANOVA)

Vamos realizar o teste H de Kruskal-Wallis para verificar se existem diferenças significativas nas medianas (ou distribuições) da elevação média (`Elevation_mean_catch`) entre as oito bacias hidrográficas mais amostradas, sem assumir normalidade dos dados.

**Hipótese Nula (H0):** As medianas da elevação média (`Elevation_mean_catch`) são as mesmas em todas as oito bacias mais amostradas (ou as amostras vêm da mesma população ou populações com a mesma distribuição de ranques).

**Hipótese Alternativa (H1):** Pelo menos uma das medianas da elevação média é diferente.

In [ ]:
from scipy import stats

# Assuming 'most_sampled_catchments' and 'df_filtered' are already defined from the previous exercise
# Ensure catchment_elevation_data is prepared correctly (same as for ANOVA)
catchment_elevation_data_kruskal = []
for catchment in most_sampled_catchments:
    catchment_elevation_data_kruskal.append(df_filtered[df_filtered['Catchment_name'] == catchment]['Elevation_mean_catch'].dropna())

# Perform Kruskal-Wallis H-test
h_statistic, p_value_kruskal = stats.kruskal(*catchment_elevation_data_kruskal)

print(f"--- Kruskal-Wallis H-test Results ---")
print(f"H-statistic: {h_statistic:.3f}")
print(f"P-value: {p_value_kruskal:.3e}")

alpha = 0.05
if p_value_kruskal < alpha:
    print(f"The P-value ({p_value_kruskal:.3e}) is less than alpha ({alpha}). We reject the null hypothesis.")
    print("There are significant differences in the median elevation among the eight most sampled catchments.")
else:
    print(f"The P-value ({p_value_kruskal:.3e}) is greater than or equal to alpha ({alpha}). We fail to reject the null hypothesis.")
    print("There are no significant differences in the median elevation among the eight most sampled catchments.")

print("\n--- Comparison with ANOVA Test ---")
print(f"ANOVA P-value: {p_value_anova:.3e}")
print(f"Kruskal-Wallis P-value: {p_value_kruskal:.3e}")

if p_value_anova < alpha and p_value_kruskal < alpha:
    print("Both ANOVA and Kruskal-Wallis tests indicate significant differences in elevation among the catchments.")
    print("This suggests a robust finding that at least one catchment's elevation distribution is different from the others.")
elif p_value_anova < alpha and p_value_kruskal >= alpha:
    print("ANOVA found significant differences, but Kruskal-Wallis did not.")
    print("This might suggest that the normality assumption for ANOVA was violated, or that the differences are in means but not necessarily in medians or ranks.")
elif p_value_anova >= alpha and p_value_kruskal < alpha:
    print("Kruskal-Wallis found significant differences, but ANOVA did not.")
    print("This could happen if the data is not normally distributed, and the Kruskal-Wallis test, being non-parametric, is more appropriate.")
else:
    print("Neither ANOVA nor Kruskal-Wallis found significant differences.")

--- Kruskal-Wallis H-test Results ---
H-statistic: 1335.373
P-value: 3.706e-284
The P-value (3.706e-284) is less than alpha (0.05). We reject the null hypothesis.
There are significant differences in the median elevation among the eight most sampled catchments.

--- Comparison with ANOVA Test ---
ANOVA P-value: 1.370e-285
Kruskal-Wallis P-value: 3.706e-284
Both ANOVA and Kruskal-Wallis tests indicate significant differences in elevation among the catchments.
This suggests a robust finding that at least one catchment's elevation distribution is different from the others.
